# 03: 标准化 + log1p + 高可变基因 HVG

在过滤后的 counts 矩阵上执行标准 scanpy 标准化管线：

1. **保留原始 counts** 到 `adata.layers['counts']`——下游方法（scVI、scANVI、DESeq2、
   pseudobulk）需要原始计数数据。这是对 sparse CSR 矩阵的拷贝。
2. **文库大小标准化**到 10000 counts/细胞。
   **为什么是 1e4？** 这是 scRNA-seq 领域的标准约定（scanpy 默认值），
   目的是消除不同细胞间测序深度的差异，使基因表达量可比。
3. **Log1p 变换**做方差稳定化。
   **为什么 log1p？** 单细胞计数数据呈离散、右偏分布。log(1+x) 变换后
   接近正态分布，适合 PCA 等线性方法。加 1 避免 log(0) 的问题。
4. **高可变基因 HVG 选择**——识别变化最大的基因用于降维。
   **为什么只保留 HVG？** 大多数基因在所有细胞中表达量都差不多（housekeeping genes），
   不携带区分细胞类型的信息。保留前 2000 个 HVG 大幅降噪、节省内存，
   且几乎不损失生物学信号。
5. **Float32 转换**——Memory Discipline #5，相比 float64 内存减半，
   对单细胞数据无有意义精度损失。

**本 notebook 产出**：
- `adata.layers['counts']`——原始 counts，为下游方法保留
- `adata.X`——标准化+log1p变换后的 float32
- `adata.var['highly_variable']`——HVG 布尔掩码
- `adata.uns['normalize_v1']`——记录标准化参数
- 03 checkpoint `.h5ad` 文件，供 04 嵌入使用

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：02（QC + 过滤），读 `02_qcd_v*.h5ad`
- **下游**：04（多方法嵌入），产出 `03_normalized_v*.h5ad`

### 为什么要迭代回跑？
单细胞分析不是一次性跑完就结束的管线。在后续 04（看 UMAP 嵌入效果）、
05（看 Leiden 分群是否合理）、06（细胞注释时发现误分群）的过程中，
都可能发现需要回到这里调整 HVG 数量（`N_TOP_GENES`）或 HVG 选择方法（`HVG_FLAVOR`）。

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `nancang_02_qcd_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `nancang_03_normalized_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `N_TOP_GENES` 或 `HVG_FLAVOR`）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为参数合理、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：04 的 `UPSTREAM_PATH` 指向你决定采用的 03 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"03_normalized"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"有哪些 03 版本？哪些依赖 03_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  — 02 产出文件路径。
#                   如需回跑：指向你要复用的上游版本（如 02_qcd_v2.h5ad）。
# OUTPUT_PATH    — 本 stage 产出 checkpoint 路径。
#                   版本号 _v1 与 adata.uns["version"] 保持一致。
#                   如需回跑：bump 版本号 _v1→_v2，旧版不覆盖。
# N_TOP_GENES    — HVG 数量（默认 2000）。调大 = 更多基因参与降维，
#                   可能捕获更细微的生物学信号但噪声也更多。
# HVG_FLAVOR     — HVG 选择方法。"seurat" 用于 log-normalized 数据；
#                   "seurat_v3" 用于原始 counts（需配合 layers['counts']）。

UPSTREAM_PATH = "results/nancang_02_qcd_v1.h5ad"
OUTPUT_PATH   = "results/nancang_03_normalized_v1.h5ad"

N_TOP_GENES   = 2000    # 高可变基因数量，每次调参后 bump OUTPUT_PATH 版本号
HVG_FLAVOR    = "seurat"  # "seurat" | "seurat_v3" | "cell_ranger"

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")


In [ ]:
# 导入模块并加载 02 产出。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import datetime

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

print("Loading upstream:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

In [ ]:
# 将原始 counts 保留到 adata.layers['counts']——下游方法（scVI、scANVI、
# DESeq2、pseudobulk）需要原始计数数据，而非标准化后的数据。
# 这是 sparse CSR 矩阵的显式拷贝——此位置内存翻倍是预期行为，
# 拷贝操作显式可追踪，便于后续审计。
print("Copying raw counts to adata.layers['counts']...")
adata.layers['counts'] = adata.X.copy()
print(f"layers keys: {list(adata.layers.keys())}")

In [ ]:
# 文库大小标准化：将每个细胞的总 counts 缩放到 10,000。
# 这是 scRNA-seq 领域的标准操作，目的是消除不同细胞间测序深度差异，
# 使基因表达量在不同细胞间可比。
print("\nNormalizing total counts per cell (target_sum=1e4)...")
sc.pp.normalize_total(adata, target_sum=1e4)
print(f"After normalize_total: X mean={adata.X.mean():.2f}, max={adata.X.max():.0f}")

In [ ]:
# Log1p 变换：log(1+x) 做方差稳定化。
# 将离散、右偏的计数数据转换为近似正态分布，
# 使其适合 PCA 等基于线性假设的下游方法。
# 加 1 避免 log(0) 的数值问题。
print("\nApplying log1p transform...")
sc.pp.log1p(adata)
print(f"After log1p: X mean={adata.X.mean():.4f}")

In [ ]:
# 高可变基因（HVG）选择——识别变化最大的基因用于降维。
# 大多数基因在所有细胞中表达量相近（housekeeping genes），不携带区分细胞类型的信息。
# 只保留前 N_TOP_GENES 个 HVG 可大幅降噪、节省内存，且几乎不损失生物学信号。
print(f"\nSelecting top {N_TOP_GENES} HVGs (flavor={HVG_FLAVOR})...")
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_TOP_GENES,
    flavor=HVG_FLAVOR,
)
n_hvg = adata.var['highly_variable'].sum()
print(f"HVGs selected: {n_hvg}")

In [ ]:
# 转为 float32——内存纪律 #5（Memory Discipline #5）。
# 单细胞标准化 counts 和嵌入携带的信息精度远低于 float64 的 15 位有效数字——
# float64 只是白白浪费一倍内存，对分析结果无实质帮助。
print("\nCasting adata.X to float32 (Memory Discipline #5)...")
adata.X = adata.X.astype(np.float32)
print(f"X dtype now: {adata.X.dtype}")

In [ ]:
# HVG 诊断图：平均表达量 vs 离散度，HVG 高亮标注。
# PI 检查此图确认 HVG 选择捕获了预期的高表达变异基因，
# 且未过度选择低表达噪声基因。
sc.pl.highly_variable_genes(adata, save=".png")
# sc.pl.highly_variable_genes 在 scanpy >=1.10 中使用独立 figure；
# 保存的图在 ./figures/highly_variable_genes.png
import os, shutil
fig_src = "figures/highly_variable_genes.png"
fig_dst = "results/figures/03_hvg.png"
if os.path.exists(fig_src):
    shutil.move(fig_src, fig_dst)
    print(f"HVG plot saved to {fig_dst}")


In [ ]:
# 记录标准化参数——plain adata.uns 写入（SPEC Run Metadata 规范）。
# 版本化键（normalize_v1）允许后续用不同参数重跑时共存，
# 便于对比不同参数的效果。
adata.uns['normalize_v1'] = {
    "target_sum":  1e4,
    "log_transformed": True,
    "hvg_flavor":  HVG_FLAVOR,
    "n_top_genes": N_TOP_GENES,
    "n_hvg":       n_hvg,
    "timestamp":   datetime.datetime.now().isoformat(),
}

# 同时记录 layer 的来源信息，便于追溯。
adata.uns["counts_layer"] = {
    "name": "counts",
    "description": "Raw counts preserved from 02 post-filter adata.X (before normalize_total)",
    "source": UPSTREAM_PATH,
}


# 统一追踪字段（SPEC 规范——供迭代回跑追溯链使用）
# stage/status/upstream/version 四个字段在 pipeline 编号间保持一致命名。
adata.uns["stage"] = "03_normalized"   # 本 stage 标识
adata.uns["status"] = "experimental"        # experimental / promoted（PI 审查后手改 promoted）
adata.uns["upstream"] = [UPSTREAM_PATH]       # 上游文件，完整溯源链
adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致
print("normalize_v1:", adata.uns['normalize_v1'])

In [ ]:
# 内存纪律自检——写入前一次断言（SPEC Memory Discipline 规范）。
# 守卫最高影响的内存退化：adata.X 变 dense 或丢失 float32。
# 如果此处失败，逐 cell 排查是哪个上游操作 densify 或 cast 了矩阵。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# 将本 stage 产出 checkpoint 写出到磁盘。
# compression="lzf" 是内存纪律 #4——比 gzip 快、
# 比不压缩约小 30%，且保留 sparse CSR 结构。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

# 验证文件已正确写出且可读。
import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 stage 边界释放内存（内存纪律 #3）。
# 不释放的话 Jupyter kernel 会一直持有上一 stage 的 AnnData，
# 后续 stage 在同一 kernel 中累积导致 OOM。
del adata
import gc
gc.collect()
print("Memory released.")